# Case study: forecasting monthly Airbnb demand in Berlin

*Session 12 practice (all three blocks). Author: course team, licence CC-BY-4.0. Data: Inside Airbnb, Berlin, snapshot of 26 June 2026 (CC BY 4.0).*

**Question.** How many Airbnb stays will there be in Berlin per month over the next twelve months, and how certain is that forecast? We measure stays by their proxy, the number of guest reviews per month. A host, a cleaning company or a city office could use such a forecast to plan prices, staff or monitoring.

- **Part 1 (block 1):** build the monthly series from `reviews_monthly.parquet`, check its quality, describe it and compute the baselines.
- **Part 2 (block 2):** fit exponential smoothing with a prediction interval, decide how to treat the pandemic, and compare with the baselines and the airline ARIMA model.
- **Part 3 (block 3):** add a lag-feature model, backtest all models from six rolling origins, and recommend one with its prediction interval.
- **Part 4 (optional):** the same comparison for the twelve districts: many smaller series instead of one.

Theory: [01](../theory/01-time-series-and-baselines.md), [02](../theory/02-exponential-smoothing-and-arima.md), [03](../theory/03-lag-features-and-backtesting.md). Prepare the data once with `uv run python case-study/prepare_airbnb.py`.

*Ethics:* the data are scraped from public listing pages. Work with aggregates; do not single out hosts.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")     # statsmodels convergence notes
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 12)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data" / "airbnb"

# Part 1: the series and its baselines

## 1.1 From the review table to a monthly series

`reviews_monthly` has one row per listing and month with at least one review.

In [ ]:
reviews = pd.read_parquet(DATA / "reviews_monthly.parquet")
print(reviews.head(3).to_string())
full = reviews.groupby("month")["n_reviews"].sum()
print("first and last months:", full.index.min().date(), full.index.max().date())
full.groupby(full.index.year).sum().rename("reviews per year").to_frame().T

**Questions.** (a) Look at the last two months (`full.tail()`). The snapshot was scraped between 26 June and 3 July 2026: why must June and July 2026 be dropped? (b) The table only contains listings that still existed in June 2026. What does that do to the counts of 2016 compared with 2025?

In [ ]:
y = full["2016":"2026-05"].rename("reviews")
y.index.freq = "MS"
log_y = np.log(y)
ax = y.plot(figsize=(10, 3.5), title="Airbnb reviews per month, Berlin")
ax.set_ylabel("reviews")
print(len(y), "months;", y.loc[["2019-07-01", "2020-04-01", "2025-07-01"]].to_dict())

## 1.2 Trend, seasonality, autocorrelation

In [ ]:
stl = STL(log_y, period=12, seasonal=13, robust=True).fit()
stl.plot()
plt.tight_layout()
recent = stl.seasonal["2022":"2025"]
np.exp(recent).groupby(recent.index.month).mean().round(2).rename("seasonal factor 2022-2025").to_frame().T

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
plot_acf(log_y, lags=24, ax=a1, title="ACF of the log counts 2016-2026")
plot_acf(log_y["2022":].diff().dropna(), lags=24, ax=a2, title="ACF of monthly growth since 2022")
plt.tight_layout()

**Question.** Which months are high, which low? How large is the pandemic dip in the remainder? Why is the left ACF high at every lag?

## 1.3 Baselines on the hold-out year (June 2025 to May 2026)

In [ ]:
train, test = y[:"2025-05"], y["2025-06":]
H = len(test)
covid = pd.Series((y.index >= "2020-03-01") & (y.index <= "2021-12-01"), index=y.index)


def mae(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


def mase(actual, forecast, y_tr, m=12):
    history = y_tr["2022":].to_numpy()                 # scale without the pandemic years
    return mae(actual, forecast) / np.mean(np.abs(history[m:] - history[:-m]))


def baselines(y_tr, index):
    last_year = y_tr.iloc[-12:].to_numpy()[:len(index)]
    return {
        "naive": pd.Series(y_tr.iloc[-1], index=index),
        "seasonal naive": pd.Series(last_year, index=index),
        "moving average (3)": pd.Series(y_tr.iloc[-3:].mean(), index=index),
        "seasonal naive x growth": pd.Series(last_year * y_tr.iloc[-12:].sum() / y_tr.iloc[-24:-12].sum(), index=index),
    }


base = baselines(train, test.index)
ax = y["2022":].plot(figsize=(10, 3.5), color="black", label="actual")
for name, fc in base.items():
    fc.plot(ax=ax, ls="--", label=name)
ax.legend(ncol=2)
pd.DataFrame({name: {"MAE": mae(test, fc), "MASE": mase(test, fc, train)} for name, fc in base.items()}).T.round(2)

# Part 2: exponential smoothing with a prediction interval

We fit an ETS model on the log counts: additive errors, a damped additive trend and additive seasonality (on the log scale, i.e. multiplicative on the original scale). The key decision is the history: all months since 2016, or only the months after the pandemic?

In [ ]:
def fit_ets(y_tr, start="2022"):
    return ETSModel(np.log(y_tr[start:]), error="add", trend="add", damped_trend=True,
                    seasonal="add", seasonal_periods=12).fit(disp=False)


for start in ("2016", "2022"):
    m = fit_ets(train, start)
    print(start, dict(zip(m.param_names[:4], m.params[:4].round(3))), f"MAE {mae(test, np.exp(m.forecast(H))):.0f}")

In [ ]:
ets = fit_ets(train)
frame = np.exp(ets.get_prediction(start=test.index[0], end=test.index[-1]).summary_frame(alpha=0.05))
ax = y["2022":].plot(figsize=(10, 3.5), color="black", label="actual")
ax.fill_between(frame.index, frame["pi_lower"], frame["pi_upper"], alpha=0.2, label="95 % interval")
frame["mean"].plot(ax=ax, ls="--", label="ETS since 2022")
ax.legend(loc="upper left")
inside = ((test >= frame["pi_lower"]) & (test <= frame["pi_upper"])).sum()
print(f"ETS MAE {mae(test, frame['mean']):.0f}, MASE {mase(test, frame['mean'], train):.2f}, {inside} of {H} months inside the interval")

## 2.2 The airline model with the pandemic marked as missing

The state-space form of ARIMA skips missing months. We mark March 2020 to December 2021 as missing and keep the years before and after.

In [ ]:
def fit_airline(y_tr):
    log_gap = np.log(y_tr).where(~covid[y_tr.index])
    return SARIMAX(log_gap, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)


fc = fit_airline(train).get_forecast(H)
air = np.exp(fc.predicted_mean)
ci = np.exp(fc.conf_int(alpha=0.05))
inside = ((test >= ci.iloc[:, 0]) & (test <= ci.iloc[:, 1])).sum()
print(f"airline MAE {mae(test, air):.0f}, MASE {mase(test, air, train):.2f}, {inside} of {H} months inside the interval")

**Exercise 2.1.** Fit ETS since July 2021 and since January 2023 instead of January 2022. How much do MAE and the interval change? What does this tell you about the reliability of a model fitted on a short history?

**Exercise 2.2.** Fit the airline model *without* marking the pandemic months as missing. Compare MAE and interval width.

In [ ]:
# Exercises 2.1 and 2.2: your code here

# Part 3: lag features, backtest and recommendation

## 3.1 A lag-feature model (direct, 12 months ahead)

Target: the yearly growth on the log scale (log y_t − log y_{t−12}), so that a tree can forecast values above the training range. Features: lag 12, lag 24 and the month, all known twelve months ahead. Rows touched by the pandemic (target, lag 12 or lag 24 between March 2020 and December 2021) are dropped.

In [ ]:
table = pd.DataFrame({"target": log_y - log_y.shift(12), "lag12": log_y.shift(12),
                      "lag24": log_y.shift(24), "month": y.index.month})
touched = covid | covid.shift(12, fill_value=False) | covid.shift(24, fill_value=False)
table = table[~touched].dropna()


def fit_lag_model(y_tr):
    rows = table.loc[:y_tr.index[-1]]
    return HistGradientBoostingRegressor(min_samples_leaf=5, random_state=0).fit(
        rows.drop(columns="target"), rows["target"])


def predict_lag_model(model, index):
    return np.exp(table.loc[index, "lag12"] + model.predict(table.loc[index].drop(columns="target")))


lag_fc = predict_lag_model(fit_lag_model(train), test.index)
print(len(table.loc[:"2025-05"]), "training rows;", f"lag model MAE {mae(test, lag_fc):.0f}, MASE {mase(test, lag_fc, train):.2f}")

## 3.2 Rolling-origin backtest

Six folds: the origin moves forward three months at a time, from March 2024 to June 2025; each test period has twelve months. The last fold is the hold-out year.

In [ ]:
rows = []
for origin in pd.date_range("2024-03-01", "2025-06-01", freq="3MS"):
    test_index = pd.date_range(origin, periods=12, freq="MS")
    y_tr, y_te = y[:origin - pd.offsets.MonthBegin()], y[test_index]
    fcs = baselines(y_tr, test_index)
    pred = np.exp(fit_ets(y_tr).get_prediction(start=test_index[0], end=test_index[-1]).summary_frame(alpha=0.05))
    fcs["ETS"] = pred["mean"]
    a = fit_airline(y_tr).get_forecast(12)
    a_ci = np.exp(a.conf_int(alpha=0.05))
    fcs["airline"] = np.exp(a.predicted_mean)
    fcs["lag model"] = predict_lag_model(fit_lag_model(y_tr), test_index)
    cover = {"ETS": ((y_te >= pred["pi_lower"]) & (y_te <= pred["pi_upper"])).mean(),
             "airline": ((y_te.to_numpy() >= a_ci.iloc[:, 0].to_numpy()) & (y_te.to_numpy() <= a_ci.iloc[:, 1].to_numpy())).mean()}
    for name, fc in fcs.items():
        rows.append({"test from": origin.strftime("%Y-%m"), "model": name, "MAE": mae(y_te, fc),
                     "MASE": mase(y_te, fc, y_tr), "coverage": cover.get(name, np.nan)})
backtest = pd.DataFrame(rows)
backtest.pivot(index="test from", columns="model", values="MASE").round(2)

In [ ]:
summary = backtest.groupby("model").agg(MAE=("MAE", "mean"), MASE=("MASE", "mean"),
                                        MASE_worst=("MASE", "max"), coverage=("coverage", "mean"))
summary["folds best"] = (backtest.loc[backtest.groupby("test from")["MASE"].idxmin(), "model"]
                         .value_counts().reindex(summary.index).fillna(0).astype(int))
summary.sort_values("MASE").round(2)

## 3.3 Recommendation

Write your recommendation for a reader who plans with the forecast, for example the tourism office or a cleaning company (4–6 sentences): which model, its forecast for the next twelve months with the 95 % interval, how accurate it was in the backtest compared with the best simple rule, and what could make the forecast wrong (reviews as a proxy for stays, listings missing from the snapshot, the unusually high May 2026, the EU registration rules that apply since 20 May 2026).

The cell below refits the airline model on all data up to May 2026 and forecasts June 2026 to May 2027.

In [ ]:
final = fit_airline(y)
fc = final.get_forecast(12)
outlook = pd.DataFrame({"mean": np.exp(fc.predicted_mean)}).join(np.exp(fc.conf_int(alpha=0.05)))
outlook.columns = ["mean", "pi_lower", "pi_upper"]
ax = y["2022":].plot(figsize=(10, 3.5), color="black", label="actual")
ax.fill_between(outlook.index, outlook["pi_lower"], outlook["pi_upper"], alpha=0.2, label="95 % interval")
outlook["mean"].plot(ax=ax, ls="--", label="airline forecast")
ax.legend(loc="upper left")
print(f"expected total June 2026 to May 2027: {outlook['mean'].sum():.0f} (last 12 months: {y['2025-06':].sum()})")
outlook.round(0)

*Your recommendation:* ...

# Part 4 (optional): many smaller series, one per district

A district office plans for its own area. We join each listing's district and repeat the hold-out comparison for the twelve districts: does the ranking of the models hold for smaller, noisier series?

In [ ]:
listings = pd.read_parquet(DATA / "listings.parquet", columns=["id", "district"])
by_district = reviews.merge(listings, left_on="listing_id", right_on="id")
months = pd.date_range("2016-01-01", "2026-05-01", freq="MS")
rows = []
for district, part in by_district.groupby("district"):
    s = part.groupby("month")["n_reviews"].sum().reindex(months, fill_value=0).astype(float)   # keep empty months
    s.index.freq = "MS"
    s_tr, s_te = s[:"2025-05"], s["2025-06":]
    b = baselines(s_tr, s_te.index)
    ets_d = np.exp(ETSModel(np.log1p(s_tr["2022":]), error="add", trend="add", damped_trend=True,
                            seasonal="add", seasonal_periods=12).fit(disp=False).forecast(12)) - 1
    log_gap = np.log1p(s_tr).where(~covid[s_tr.index])
    air_d = np.exp(SARIMAX(log_gap, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False).forecast(12)) - 1
    rows.append({"district": district, "mean per month": round(s_te.mean()),
                 "MASE seasonal naive x growth": mase(s_te, b["seasonal naive x growth"], s_tr),
                 "MASE ETS": mase(s_te, ets_d, s_tr), "MASE airline": mase(s_te, air_d, s_tr)})
pd.DataFrame(rows).sort_values("mean per month", ascending=False).round(2)

## Exercises

1. Repeat the backtest with origins every month instead of every three months. Do the rankings change? Why are the folds now strongly correlated?
2. Add a lockdown dummy instead of missing values: `SARIMAX(np.log(y_tr), exog=covid[y_tr.index].astype(float), ...)`. What does the coefficient say, and how do the forecasts compare? (Forecasting needs the future values of the dummy: zeros.)
3. Use the `calendar` table: compute the share of nights still bookable per month for the next twelve months. Why is this not a forecast of demand?
4. Optional: repeat the backtest with StatsForecast (`AutoETS`, `AutoARIMA`, `SeasonalNaive`, `cross_validation` with `step_size=3`), as in workbook 08, and compare the results.
5. Optional: the workbook [11-optional-ebti-decision-forecast.ipynb](11-optional-ebti-decision-forecast.ipynb) applies the same workflow to the monthly BTI decisions of the main case study. Compare the two series: which has the stronger season, which the larger break?

In [ ]:
# Exercise 1: your code here